# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access dataset metadata
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields (by @id)
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    # List fields in this recordset
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Extract data from all available record sets

# Build a list of record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set '{rs_id}' with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Show head of one record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nColumns for {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, choose the first record set and a numeric field

record_set_id = example_rs_id
df = dataframes[record_set_id]

# List all columns and guess the numeric one by typical names
numeric_field_candidates = [c for c in df.columns if any(keyword in c.lower() for keyword in ["abundance", "count", "coverage", "mw", "weight", "value"])]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    # Fallback to the first column
    numeric_field = df.columns[0]
print(f"Using '{numeric_field}' as numeric field.")

# Ensure numeric type
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example filter
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (75th percentile):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by another field (e.g. 'description', or another string column that is not numeric_field)
group_field_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If there's a suitable group_field, plot mean by group (top 10)
if 'group_field' in locals():
    top_grouped = grouped_df.sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=top_grouped.index, y=top_grouped.values)
    plt.title(f'Mean {numeric_field} by {group_field} (top 10)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR²-compliant mass spectrometry dataset describing protein abundance, molecular weights, and modifications in extracellular vesicles from human mast cells. Using `mlcroissant`, we explored the schema, listed and extracted records from each record set, and performed basic filtering, normalization, grouping, and visualization. Further statistical or domain-specific analyses can proceed using the extracted DataFrames and schema-aware references to the dataset fields and record sets by their `@id`.
